<a href="https://colab.research.google.com/github/pradeeshselvan86-rgb/ITA0616---MACHINE-LEARNING/blob/main/Candidate_Elimination_algorithm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 1. Create a Sample Dataset CSV
We will create a sample CSV file containing training examples (e.g., the standard EnjoySport dataset) to test the Candidate-Elimination algorithm.

In [1]:
import csv
import pandas as pd

# Define standard EnjoySport training data
data = [
    ['Sunny', 'Warm', 'Normal', 'Strong', 'Warm', 'Same', 'Yes'],
    ['Sunny', 'Warm', 'High', 'Strong', 'Warm', 'Same', 'Yes'],
    ['Rainy', 'Cold', 'High', 'Strong', 'Warm', 'Change', 'No'],
    ['Sunny', 'Warm', 'High', 'Strong', 'Cool', 'Change', 'Yes']
]

columns = ['Sky', 'AirTemp', 'Humidity', 'Wind', 'Water', 'Forecast', 'EnjoySport']

# Save to a CSV file
csv_file_path = 'training_data.csv'
with open(csv_file_path, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(columns)
    writer.writerows(data)

# Display the created dataset
df = pd.read_csv(csv_file_path)
display(df)

,Sky,AirTemp,Humidity,Wind,Water,Forecast,EnjoySport
0,Sunny,Warm,Normal,Strong,Warm,Same,Yes
1,Sunny,Warm,High,Strong,Warm,Same,Yes
2,Rainy,Cold,High,Strong,Warm,Change,No
3,Sunny,Warm,High,Strong,Cool,Change,Yes


### 2. Implement and Demonstrate the Candidate-Elimination Algorithm
This algorithm computes the specific ($S$) and general ($G$) boundaries of the version space.

In [2]:
import numpy as np

def candidate_elimination(csv_path):
    # Load dataset
    with open(csv_path, 'r') as f:
        reader = csv.reader(f)
        dataset = list(reader)

    headers = dataset[0]
    concepts = np.array([row[:-1] for row in dataset[1:]])
    target = np.array([row[-1] for row in dataset[1:]])

    num_attributes = len(concepts[0])

    # Initialize Specific (S) and General (G) hypotheses
    # S is initialized to the most specific hypothesis (all phi)
    # G is initialized to the most general hypothesis (all ?)
    specific_h = concepts[0].copy()
    general_h = [['?' for _ in range(num_attributes)] for _ in range(num_attributes)]

    print("Initialization:")
    print(f"S0: {specific_h}")
    print(f"G0: {general_h}\n")

    for i, (instance, label) in enumerate(zip(concepts, target)):
        print(f"Instance {i+1}: {instance} -> Target: {label}")

        if label == 'Yes':
            # For positive examples, generalize S
            for x in range(num_attributes):
                if instance[x] != specific_h[x]:
                    specific_h[x] = '?'
                    # Make sure generalized elements in G are kept consistent
                    for g in general_h:
                        if g[x] != '?' and g[x] != instance[x]:
                            general_h.remove(g)
        else:
            # For negative examples, specialize G
            for x in range(num_attributes):
                if instance[x] != specific_h[x]:
                    # Create a specialized hypothesis for each differing attribute
                    g_temp = ['?' for _ in range(num_attributes)]
                    g_temp[x] = specific_h[x]
                    if g_temp not in general_h:
                        general_h.append(g_temp)

            # Remove hypotheses from G that are too specific or inconsistent with positive cases
            general_h = [g for g in general_h if any(g[x] == '?' or g[x] == specific_h[x] for x in range(num_attributes))]

            # Clean up the general hypotheses list to remove redundant/overlapping hypotheses
            general_h = [g for g in general_h if g != ['?' for _ in range(num_attributes)]]

        print(f"S{i+1}: {specific_h}")
        print(f"G{i+1}: {general_h}\n")

    # Final Clean-up: filter out general hypotheses that do not align with S
    final_g = []
    for g in general_h:
        consistent = True
        for x in range(num_attributes):
            if g[x] != '?' and g[x] != specific_h[x]:
                consistent = False
                break
        if consistent:
            final_g.append(g)

    return specific_h, final_g

# Run Candidate Elimination
final_s, final_g = candidate_elimination('training_data.csv')

print("-" * 50)
print("FINAL RESULTS:")
print(f"Specific Boundary (S): {final_s}")
print(f"General Boundary (G): {final_g}")

Initialization:
S0: ['Sunny' 'Warm' 'Normal' 'Strong' 'Warm' 'Same']
G0: [['?', '?', '?', '?', '?', '?'], ['?', '?', '?', '?', '?', '?'], ['?', '?', '?', '?', '?', '?'], ['?', '?', '?', '?', '?', '?'], ['?', '?', '?', '?', '?', '?'], ['?', '?', '?', '?', '?', '?']]

Instance 1: ['Sunny' 'Warm' 'Normal' 'Strong' 'Warm' 'Same'] -> Target: Yes
S1: ['Sunny' 'Warm' 'Normal' 'Strong' 'Warm' 'Same']
G1: [['?', '?', '?', '?', '?', '?'], ['?', '?', '?', '?', '?', '?'], ['?', '?', '?', '?', '?', '?'], ['?', '?', '?', '?', '?', '?'], ['?', '?', '?', '?', '?', '?'], ['?', '?', '?', '?', '?', '?']]

Instance 2: ['Sunny' 'Warm' 'High' 'Strong' 'Warm' 'Same'] -> Target: Yes
S2: ['Sunny' 'Warm' '?' 'Strong' 'Warm' 'Same']
G2: [['?', '?', '?', '?', '?', '?'], ['?', '?', '?', '?', '?', '?'], ['?', '?', '?', '?', '?', '?'], ['?', '?', '?', '?', '?', '?'], ['?', '?', '?', '?', '?', '?'], ['?', '?', '?', '?', '?', '?']]

Instance 3: ['Rainy' 'Cold' 'High' 'Strong' 'Warm' 'Change'] -> Target: No
S3: ['Sunny